# This notebook documents the logic I used to create new "Features"
### Features are predictor variables that I believe could potentially predict a job failure.
### I came up with ideas for these features by looking at several example job CPU/GPU logs.

In [40]:
# Import Libraries and Connect to AWS S3
import s3fs
fs = s3fs.S3FileSystem(anon=True)
import pandas as pd
pd.set_option('display.max_columns', None)
import ast
import duckdb
import re
import numpy as np

In [41]:
# Load the Slurm Log
df_slurm = pd.read_csv('s3://mit-supercloud-dataset/datacenter-challenge/202201/slurm-log.csv')
print(df_slurm.shape)

(395914, 29)


#
# Loading a good example job to test the new features I want to add:

In [42]:
#Example Job: 1 CPU step, 1 GPU Log, Job Successful (State = 3)
job = df_slurm.loc[df_slurm['id_job'] == 4391237494359].copy()
job

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type
236300,4391237494359,16618712154521,4294967294,5042570016904,61026541062099,1,['r8937440-n43543'],20,0,0,NaN,0,0,\N,8,9223372036854784308,normal,10003,3,525600,1623428400,1623428400,1623532270,1623575066,0,0,"1=20,2=170000,4=1,5=20,1002=1","1=20,2=170000,4=1,5=20,1002=1",OTHER


In [43]:
# Gathering the CPU Logs for this job by searching the MIT Supercloud directory for files with a matching job ID
job_id = '4391237494359'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/cpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} total CPU log file(s) for Job ID {job_id}.")
if files:
    summary_count = 1
    ts_count = 1
    for file_path in files:
        full_path = f"s3://{file_path}"
        if 'summary' in file_path.lower():
            var_name = f"job_{job_id}_CPUSummary_{summary_count}"
            summary_count += 1
        else:
            var_name = f"job_{job_id}_CPUTimeSeries_{ts_count}"
            ts_count += 1
        globals()[var_name] = pd.read_csv(full_path)
        print(f"Created variable: {var_name}")
else:
    print(f"No .csv files found for Job ID {job_id}.")

Found 2 total CPU log file(s) for Job ID 4391237494359.
Created variable: job_4391237494359_CPUSummary_1
Created variable: job_4391237494359_CPUTimeSeries_1


In [44]:
# Print the CPU Summary file
job_4391237494359_CPUSummary_1

,Step,Node,Series,ElapsedTime,Min_EpochTime,Max_EpochTime,Sum_EpochTime,Avg_EpochTime,Min_CPUFrequency,Max_CPUFrequency,Sum_CPUFrequency,Avg_CPUFrequency,Min_CPUTime,Max_CPUTime,Sum_CPUTime,Avg_CPUTime,Min_CPUUtilization,Max_CPUUtilization,Sum_CPUUtilization,Avg_CPUUtilization,Min_RSS,Max_RSS,Sum_RSS,Avg_RSS,Min_VMSize,Max_VMSize,Sum_VMSize,Avg_VMSize,Min_Pages,Max_Pages,Sum_Pages,Avg_Pages,Min_ReadMB,Max_ReadMB,Sum_ReadMB,Avg_ReadMB,Min_WriteMB,Max_WriteMB,Sum_WriteMB,Avg_WriteMB
0,-4,r8937440-n43543,0,42795,1623532271,1623575066,6950433265546,1.623554e+09,1048,1048,4486488,1048.000000,0.0,0.00,0.00,0.000000,0.0,0.0,0.0,0.000000,780,2204,3340604,7.803326e+02,7776,168348,33449628,7.813508e+03,0,0,0,0.000000,0.0,0.001858,1.858000e-03,0.00000,0.0,0.002243,0.002243,0.000001
1,batch,r8937440-n43543,0,42795,1623532271,1623575066,6950433265546,1.623554e+09,0,1226,308766,72.124737,0.0,43.76,170281.68,39.776146,0.0,437.6,1702816.8,397.761458,4748,37435948,95256591992,2.225101e+07,201088,109295496,341690272136,7.981553e+07,0,10,36395,8.501518,0.0,1086.672377,2.815067e+06,657.57224,0.0,8.128551,10.567973,0.002469


In [45]:
# Print the CPU Timeseries file
job_4391237494359_CPUTimeSeries_1.head()

,Step,Node,Series,ElapsedTime,EpochTime,CPUFrequency,CPUTime,CPUUtilization,RSS,VMSize,Pages,ReadMB,WriteMB
0,-4,r8937440-n43543,0,0,1623532271,1048,0.0,0.0,2204,168348,0,0.000000,0.000000
1,-4,r8937440-n43543,0,10,1623532281,1048,0.0,0.0,780,7776,0,0.001858,0.002243
2,-4,r8937440-n43543,0,20,1623532291,1048,0.0,0.0,780,7776,0,0.000000,0.000000
3,-4,r8937440-n43543,0,30,1623532301,1048,0.0,0.0,780,7776,0,0.000000,0.000000
4,-4,r8937440-n43543,0,40,1623532311,1048,0.0,0.0,780,7776,0,0.000000,0.000000


In [46]:
# Gathering the GPU Log for this job
job_id = '4391237494359'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/gpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} GPU log file(s) for Job ID {job_id}.")
if files:
    for i, file_path in enumerate(files, start=1):
        var_name = f"job_{job_id}_GPULog_{i}"
        globals()[var_name] = pd.read_csv(f"s3://{file_path}")
        print(f"Created variable: {var_name}")
else:
    print(f"No GPU files found for Job {job_id}.")

Found 1 GPU log file(s) for Job ID 4391237494359.
Created variable: job_4391237494359_GPULog_1


In [47]:
# Print the GPU Timeseries file
job_4391237494359_GPULog_1.head()

,timestamp,gpu_index,utilization_gpu_pct,utilization_memory_pct,memory_free_MiB,memory_used_MiB,temperature_gpu,temperature_memory,power_draw_W,pcie_link_width_current
0,1.623518e+09,0,0,0,32510,0,58,57,52.54,16
1,1.623518e+09,0,0,0,32510,0,58,57,52.73,16
2,1.623518e+09,0,0,0,32510,0,58,57,52.76,16
3,1.623518e+09,0,0,0,32510,0,58,57,52.57,16
4,1.623518e+09,0,0,0,32506,4,58,57,52.56,16


#
# Feature Engineering:

In [48]:
# Create new columns for job duration
job['duration_sec'] = job['time_end'] - job['time_start']
job['duration_hrs'] = job['duration_sec'] / 3600
job[['time_start', 'time_end', 'duration_sec', 'duration_hrs']]

,time_start,time_end,duration_sec,duration_hrs
236300,1623532270,1623575066,42796,11.887778


In [49]:
# Create new column for count of GPU's allocated for the job (using the tres_alloc field)
def get_allocated_gpu_count(tres_str):
    tres_str = str(tres_str)
    mappings = dict(re.findall(r'(\d+)=(\d+)', tres_str))
    
    # Priority 1: Check MIT Specific IDs (1001 or 1002)
    for specific_id in ['1001', '1002']:
        if specific_id in mappings:
            return int(mappings[specific_id])
    return 0

job['gpus_alloc'] = job['tres_alloc'].apply(get_allocated_gpu_count)
job[['tres_alloc', 'gpus_alloc']]

,tres_alloc,gpus_alloc
236300,"1=20,2=170000,4=1,5=20,1002=1",1


In [50]:
# Create new column for count of CPU's allocated for the job (using the tres_alloc field)
def get_allocated_cpu_count(tres_str):
    tres_str = str(tres_str)
    mappings = dict(re.findall(r"(\d+)=(\d+)", tres_str))

    # TRES ID '1' represents CPUs
    if "1" in mappings:
        return int(mappings["1"])
    return 0

job["cpus_alloc"] = job["tres_alloc"].apply(get_allocated_cpu_count)
job[["tres_alloc", "cpus_alloc"]]

,tres_alloc,cpus_alloc
236300,"1=20,2=170000,4=1,5=20,1002=1",20


In [51]:
# Create new columns for a readable start and end date
job['time_start_dt'] = pd.to_datetime(job['time_start'], unit='s')
job['time_end_dt'] = pd.to_datetime(job['time_end'], unit='s')
job[['time_start', 'time_end', 'time_start_dt', 'time_end_dt']]

,time_start,time_end,time_start_dt,time_end_dt
236300,1623532270,1623575066,2021-06-12 21:11:10,2021-06-13 09:04:26


In [52]:
# New column for count of CPU Steps
query = "SELECT COUNT(*) FROM job_4391237494359_CPUSummary_1"
row_count = duckdb.query(query).fetchone()[0]
job['Job_Steps'] = row_count
job[['Job_Steps']]

,Job_Steps
236300,2


In [53]:
# New columns for CPU Utilization statistics
# Divide by the number of CPU cores to get a standardized percentage
# The below code finds the max value of CPU Util in the duration range (0:05 - 2:00)
# Excluding the data in the first 5 minutes as this is the job "ramp-up" time

target_col = "CPUUtilization" 
query = f"""
    SELECT MAX({target_col}) as max
    FROM job_4391237494359_CPUTimeSeries_1
    WHERE CAST("Step" AS VARCHAR) = '0' OR CAST("Step" AS VARCHAR) = 'batch'
        AND "ElapsedTime" > 120 
        AND "ElapsedTime" <= 7200
"""
# Fetch the scalar result
max_val = duckdb.query(query).fetchone()[0]
job['Max_CPU_Util'] = max_val / job['cpus_alloc']
job[['Max_CPU_Util']]

,Max_CPU_Util
236300,21.775


In [54]:
# Minimum CPU Utilization

target_col = "CPUUtilization" 
query = f"""
    SELECT MIN({target_col}) as min
    FROM job_4391237494359_CPUTimeSeries_1
    WHERE CAST("Step" AS VARCHAR) = '0' OR CAST("Step" AS VARCHAR) = 'batch'
        AND "ElapsedTime" > 120 
        AND "ElapsedTime" <= 7200
"""
# Fetch the scalar result
min_val = duckdb.query(query).fetchone()[0]
job['Min_CPU_Util'] = min_val / job['cpus_alloc']
job[['Min_CPU_Util']]

,Min_CPU_Util
236300,9.425


In [55]:
# Median CPU Utilization

target_col = "CPUUtilization" 
query = f"""
    SELECT MEDIAN({target_col}) as filtered_median
    FROM job_4391237494359_CPUTimeSeries_1
    WHERE CAST("Step" AS VARCHAR) = '0' OR CAST("Step" AS VARCHAR) = 'batch'
        AND "ElapsedTime" > 120 
        AND "ElapsedTime" <= 7200
"""
median_val = duckdb.query(query).fetchone()[0]
job['Med_CPU_Util'] = median_val / job['cpus_alloc']
job[['Med_CPU_Util']]

,Med_CPU_Util
236300,20.06


In [56]:
# Inter-Quartile Range (IQR) of CPU Utilization
# This captures the variance of the data

target_col = "CPUUtilization"
query = f"""
    SELECT 
        QUANTILE_CONT({target_col}, 0.75) - QUANTILE_CONT({target_col}, 0.25) as iqr_value
    FROM job_4391237494359_CPUTimeSeries_1
    WHERE CAST("Step" AS VARCHAR) = '0' OR CAST("Step" AS VARCHAR) = 'batch'
        AND "ElapsedTime" > 120 
        AND "ElapsedTime" <= 7200
"""
iqr_result = duckdb.query(query).fetchone()[0]
job['IQR_CPU_Util'] = iqr_result / job['cpus_alloc']
job[['IQR_CPU_Util']]

,IQR_CPU_Util
236300,0.26


In [57]:
# New columns for Resident Memory Footprint Set Size (RSS) statistics
# Combining code for all three columns to make things more streamlined

target_col = "RSS"
query = f"""
        SELECT 
            MAX({target_col}),
            MIN({target_col}),
            MEDIAN({target_col}),
            QUANTILE_CONT({target_col}, 0.75) - QUANTILE_CONT({target_col}, 0.25)
        FROM job_4391237494359_CPUTimeSeries_1
        WHERE (CAST("Step" AS VARCHAR) = '0' OR CAST("Step" AS VARCHAR) = 'batch')
            AND "ElapsedTime" > 120 
            AND "ElapsedTime" <= 7200
    """
    
# Execute and fetch all three results at once
max_val, min_val, med_val, iqr_val = duckdb.query(query).fetchone()
    
# Dynamically assign to job dictionary
job[f'Max_{target_col}'] = max_val
job[f'Min_{target_col}'] = min_val
job[f'Med_{target_col}'] = med_val
job[f'IQR_{target_col}'] = iqr_val
job[['Max_RSS', 'Min_RSS', 'Med_RSS', 'IQR_RSS']]

,Max_RSS,Min_RSS,Med_RSS,IQR_RSS
236300,13854192,6522792,9555984.0,2918685.0


In [58]:
# Slope of RSS (to show whether it is increasing or decreasing)

target_col = "RSS"
query = f"""
    SELECT REGR_SLOPE({target_col}, "ElapsedTime") as cpu_slope
    FROM job_4391237494359_CPUTimeSeries_1
    WHERE CAST("Step" AS VARCHAR) = '0' OR CAST("Step" AS VARCHAR) = 'batch'
      AND "ElapsedTime" > 120 
      AND "ElapsedTime" <= 7200
"""
slope_val = duckdb.query(query).fetchone()[0]
job['RSS_Slope'] = slope_val
job[['RSS_Slope']]

,RSS_Slope
236300,793.850014


In [59]:
# Acceleration of RSS (to show how quickly it is increasing or decreasing)

target_col = "RSS"
query = f"""
    SELECT 
        REGR_SLOPE({target_col}, POWER("ElapsedTime", 2)) as quadratic_trend
    FROM job_4391237494359_CPUTimeSeries_1
    WHERE CAST("Step" AS VARCHAR) = '0' OR CAST("Step" AS VARCHAR) = 'batch'
      AND "ElapsedTime" > 120 
      AND "ElapsedTime" <= 7200
"""
acc_val = duckdb.query(query).fetchone()[0]
job['RSS_Accel'] = acc_val
job[['RSS_Accel']]

,RSS_Accel
236300,0.100824


In [60]:
# Creating new features for other useful columns in the CPU Timeseries file via a loop:

target_columns = ["VMSize", "Pages", "ReadMB", "WriteMB"] 
for col in target_columns:
    query = f"""
        SELECT 
            MAX({col}),
            MIN({col}),
            MEDIAN({col}),
            QUANTILE_CONT({col}, 0.75) - QUANTILE_CONT({col}, 0.25)
        FROM job_4391237494359_CPUTimeSeries_1
        WHERE (CAST("Step" AS VARCHAR) = '0' OR CAST("Step" AS VARCHAR) = 'batch')
            AND "ElapsedTime" > 120 
            AND "ElapsedTime" <= 7200
    """
    
    max_val, min_val, med_val, iqr_val = duckdb.query(query).fetchone()
    
    job[f'Max_{col}'] = max_val
    job[f'Min_{col}'] = min_val
    job[f'Med_{col}'] = med_val
    job[f'IQR_{col}'] = iqr_val

In [61]:
print(job.iloc[0, -16:])

Max_VMSize       71281444
Min_VMSize       58403316
Med_VMSize     62761504.0
IQR_VMSize      2490880.0
Max_Pages               7
Min_Pages               7
Med_Pages             7.0
IQR_Pages             0.0
Max_ReadMB      783.68778
Min_ReadMB     164.500101
Med_ReadMB     665.058918
IQR_ReadMB      22.329923
Max_WriteMB      0.000572
Min_WriteMB      0.000229
Med_WriteMB      0.000563
IQR_WriteMB      0.000114
Name: 236300, dtype: object


In [62]:
# VMSize Slope

target_col = "VMSize"
query = f"""
    SELECT 
        REGR_SLOPE({target_col}, "ElapsedTime") as cpu_slope
    FROM job_4391237494359_CPUTimeSeries_1
    WHERE CAST("Step" AS VARCHAR) = '0' OR CAST("Step" AS VARCHAR) = 'batch'
      AND "ElapsedTime" > 120 
      AND "ElapsedTime" <= 7200
"""
slope_val = duckdb.query(query).fetchone()[0]
job['VMSize_Slope'] = slope_val
job[['VMSize_Slope']]

,VMSize_Slope
236300,1201.469128


In [63]:
# Adding a column to the CPU Timeseries log for RSS Velocity (Change in RSS)
job_4391237494359_CPUTimeSeries_1['RSS_Velocity'] = job_4391237494359_CPUTimeSeries_1['RSS'].diff().fillna(0)

In [64]:
# Creating new columns in the Slurm Log for RSS Velocity statistics

query = f"""
     SELECT 
         MAX(RSS_Velocity),
         MIN(RSS_Velocity),
         MEDIAN(RSS_Velocity),
         QUANTILE_CONT(RSS_Velocity, 0.75) - QUANTILE_CONT(RSS_Velocity, 0.25)
     FROM job_4391237494359_CPUTimeSeries_1
     WHERE (CAST("Step" AS VARCHAR) = '0' OR CAST("Step" AS VARCHAR) = 'batch')
          AND "ElapsedTime" > 120 
          AND "ElapsedTime" <= 7200
   """

max_val, min_val, med_val, iqr_val = duckdb.query(query).fetchone()
job[f'Max_RSS_Velocity'] = max_val
job[f'Min_RSS_Velocity'] = min_val
job[f'Med_RSS_Velocity'] = med_val
job[f'IQR_RSS_Velocity'] = iqr_val
print(job.iloc[0, -4:])

Max_RSS_Velocity    2590340.0
Min_RSS_Velocity   -2575908.0
Med_RSS_Velocity       6650.0
IQR_RSS_Velocity       2052.0
Name: 236300, dtype: object


In [65]:
# Adding a column to the CPU Timeseries log for VM_RSS_Gap (difference between VMSize and RSS)
job_4391237494359_CPUTimeSeries_1['VM_RSS_Gap'] = job_4391237494359_CPUTimeSeries_1['VMSize'] - job_4391237494359_CPUTimeSeries_1['RSS']

In [66]:
# VM_RSS_Gap features

query = f"""
     SELECT 
         MAX(VM_RSS_Gap),
         MIN(VM_RSS_Gap),
         MEDIAN(VM_RSS_Gap),
         QUANTILE_CONT(VM_RSS_Gap, 0.75) - QUANTILE_CONT(VM_RSS_Gap, 0.25)
     FROM job_4391237494359_CPUTimeSeries_1
     WHERE (CAST("Step" AS VARCHAR) = '0' OR CAST("Step" AS VARCHAR) = 'batch')
          AND "ElapsedTime" > 120 
          AND "ElapsedTime" <= 7200
   """
    
max_val, min_val, med_val, iqr_val = duckdb.query(query).fetchone()
    
job[f'Max_VM_RSS_Gap'] = max_val
job[f'Min_VM_RSS_Gap'] = min_val
job[f'Med_VM_RSS_Gap'] = med_val
job[f'IQR_VM_RSS_Gap'] = iqr_val
print(job.iloc[0, -4:])

Max_VM_RSS_Gap      57427252
Min_VM_RSS_Gap      51628980
Med_VM_RSS_Gap    52373330.0
IQR_VM_RSS_Gap     1329396.0
Name: 236300, dtype: object


In [67]:
# VM_RSS_Gap_Slope

target_col = "VM_RSS_Gap"
query = f"""
    SELECT 
        REGR_SLOPE({target_col}, "ElapsedTime") as cpu_slope
    FROM job_4391237494359_CPUTimeSeries_1
    WHERE CAST("Step" AS VARCHAR) = '0' OR CAST("Step" AS VARCHAR) = 'batch'
      AND "ElapsedTime" > 120 
      AND "ElapsedTime" <= 7200
"""
slope_val = duckdb.query(query).fetchone()[0]
job['VM_RSS_Gap_Slope'] = slope_val
job[['VM_RSS_Gap_Slope']]

,VM_RSS_Gap_Slope
236300,407.619114


In [68]:
# New column in the GPU Timeseries file for seconds since start of GPU process
job_4391237494359_GPULog_1['elapsed_seconds'] = job_4391237494359_GPULog_1['timestamp'] - job_4391237494359_GPULog_1['timestamp'].min()

In [69]:
# New GPU Features

target_columns = ["utilization_gpu_pct", "utilization_memory_pct", "temperature_gpu", 
                  "temperature_memory", "power_draw_W"] 
for col in target_columns:
    query = f"""
        SELECT 
            MAX({col}),
            MIN({col}),
            MEDIAN({col}),
            QUANTILE_CONT({col}, 0.75) - QUANTILE_CONT({col}, 0.25)
        FROM job_4391237494359_GPULog_1
        WHERE "elapsed_seconds" > 120 
            AND "elapsed_seconds" <= 7200
    """
    
    max_val, min_val, med_val, iqr_val = duckdb.query(query).fetchone()
    
    job[f'Max_{col}'] = max_val
    job[f'Min_{col}'] = min_val
    job[f'Med_{col}'] = med_val
    job[f'IQR_{col}'] = iqr_val

print(job.iloc[0, -20:])

Max_utilization_gpu_pct          100
Min_utilization_gpu_pct            0
Med_utilization_gpu_pct         90.0
IQR_utilization_gpu_pct          3.0
Max_utilization_memory_pct        46
Min_utilization_memory_pct         0
Med_utilization_memory_pct      41.0
IQR_utilization_memory_pct       4.0
Max_temperature_gpu               71
Min_temperature_gpu               58
Med_temperature_gpu             69.0
IQR_temperature_gpu              1.0
Max_temperature_memory            75
Min_temperature_memory            58
Med_temperature_memory          71.0
IQR_temperature_memory           1.0
Max_power_draw_W              244.45
Min_power_draw_W                53.7
Med_power_draw_W              214.92
IQR_power_draw_W               25.38
Name: 236300, dtype: object


In [70]:
# Power_Util_Ratio
job['Power_Util_Ratio'] = (job['Med_power_draw_W'] / job['Med_utilization_gpu_pct']).replace([np.inf, -np.inf], np.nan)
job[['Power_Util_Ratio']]

,Power_Util_Ratio
236300,2.388


In [71]:
# Printing final column list
job

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,duration_sec,duration_hrs,gpus_alloc,cpus_alloc,time_start_dt,time_end_dt,Job_Steps,Max_CPU_Util,Min_CPU_Util,Med_CPU_Util,IQR_CPU_Util,Max_RSS,Min_RSS,Med_RSS,IQR_RSS,RSS_Slope,RSS_Accel,Max_VMSize,Min_VMSize,Med_VMSize,IQR_VMSize,Max_Pages,Min_Pages,Med_Pages,IQR_Pages,Max_ReadMB,Min_ReadMB,Med_ReadMB,IQR_ReadMB,Max_WriteMB,Min_WriteMB,Med_WriteMB,IQR_WriteMB,VMSize_Slope,Max_RSS_Velocity,Min_RSS_Velocity,Med_RSS_Velocity,IQR_RSS_Velocity,Max_VM_RSS_Gap,Min_VM_RSS_Gap,Med_VM_RSS_Gap,IQR_VM_RSS_Gap,VM_RSS_Gap_Slope,Max_utilization_gpu_pct,Min_utilization_gpu_pct,Med_utilization_gpu_pct,IQR_utilization_gpu_pct,Max_utilization_memory_pct,Min_utilization_memory_pct,Med_utilization_memory_pct,IQR_utilization_memory_pct,Max_temperature_gpu,Min_temperature_gpu,Med_temperature_gpu,IQR_temperature_gpu,Max_temperature_memory,Min_temperature_memory,Med_temperature_memory,IQR_temperature_memory,Max_power_draw_W,Min_power_draw_W,Med_power_draw_W,IQR_power_draw_W,Power_Util_Ratio
236300,4391237494359,16618712154521,4294967294,5042570016904,61026541062099,1,['r8937440-n43543'],20,0,0,NaN,0,0,\N,8,9223372036854784308,normal,10003,3,525600,1623428400,1623428400,1623532270,1623575066,0,0,"1=20,2=170000,4=1,5=20,1002=1","1=20,2=170000,4=1,5=20,1002=1",OTHER,42796,11.887778,1,20,2021-06-12 21:11:10,2021-06-13 09:04:26,2,21.775,9.425,20.06,0.26,13854192,6522792,9555984.0,2918685.0,793.850014,0.100824,71281444,58403316,62761504.0,2490880.0,7,7,7.0,0.0,783.68778,164.500101,665.058918,22.329923,0.000572,0.000229,0.000563,0.000114,1201.469128,2590340.0,-2575908.0,6650.0,2052.0,57427252,51628980,52373330.0,1329396.0,407.619114,100,0,90.0,3.0,46,0,41.0,4.0,71,58,69.0,1.0,75,58,71.0,1.0,244.45,53.7,214.92,25.38,2.388
